### we employed Chrome's semantic similarity search algorithm and found that it
### performed quite well in retrieving content relevant to a user's prompt.
### It did, however, come with a disadvantage if a vector store contained duplicate documents with a high
### similarity score, the method would retrieve both the original and the duplicate.
### And as we'll now discover, retrieving documents based only on the similarity metric results in a lack
### of diversity and a chance of missing important information.

## In this lesson, we'll study another retrieval algorithm that accounts for this shortcoming.

## It goes by maximal marginal relevance search or MMR search for short.



In [1]:
import config

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma  # New way
from langchain_core.documents import Document

C:\Users\moham\anaconda3\envs\Langchain_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
embedding = OpenAIEmbeddings(model="text-embedding-ada-002",
                            openai_api_key = config.api_key)

In [3]:
vectorstore = Chroma( persist_directory= './vectore representrtion folder',
                                   embedding_function= embedding
                                   )

In [4]:
vectorstore.get()

{'ids': ['2b59d12b-2b0c-469a-b517-05da0c7698ae',
  '5ee12cc8-b170-4a52-891c-b7ea881313da',
  '18b58a9b-6561-4b21-886d-a6234fbc51a6',
  '593034a0-887d-4ed3-bf70-a6dff24703db',
  'ac8557a2-625d-4c1a-a534-57dd84c6cf1f',
  'ac004c34-41be-443c-a700-5ab8c2d260ab',
  'edb9767c-4757-4078-8e1e-58944b4b4b63',
  'a7c12ab0-d2c1-48b6-b155-d106ce424907',
  '1aec6905-f407-4456-91c2-3fe88eb10223',
  '1f473953-d596-4da7-8265-df4fe1ad083e',
  '84c72abd-ae48-4dd6-814b-0f2a00fefac8',
  '462cc1fb-6a16-4808-8b52-a3a343354089',
  '716d8a6b-ab67-4b92-a8d8-f69a11554a63',
  'f7b20195-773a-4ad0-b9cd-33e2b8b46b04',
  'a2d8d7cc-0c83-4be5-b27b-ca59e88f6051',
  '536dc62b-aa6d-4aba-aa10-66af8defc5d1',
  'dc61bc5c-ff8c-4c9d-b7b8-9ee295001a85',
  '7a434846-c74d-4784-ab4a-42a9c5694793',
  'f6c087f2-eaee-44f4-9b71-5777b8aaaaf4',
  '8d549176-6080-4b0f-89b2-a2f5b367c462',
  '1fe356f5-b33a-4b86-9526-6b9acd1274b5'],
 'embeddings': None,
 'documents': ['Alright! So… Let’s discuss the not-so-obvious differences between the ter

In [5]:
added_doc = Document(page_content='''Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other
                                is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis''',
                    metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Analysis vs Analytics'}
                        )

In [6]:
vectorstore.add_documents([added_doc])

['d28f73cb-efd7-4206-a021-d472b7b225c5']

In [7]:
question = "What software do data scientists use?"

In [9]:
retrived_doc =vectorstore.similarity_search(query = question,
                                           k=3)

In [10]:
for i in retrived_doc:
    print(f"Page Contennt: {i.page_content} \n---------\n Lecture title:{i.metadata['Lecture Title']}\n")

Page Contennt: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you ne

After studying the content, we found the first document contained information on the R and Python programming

languages, which was irrelevant to the current question.

The other two documents are the original and duplicate chunks that don't reference any of the software

mentioned.

It seems like the method missed important information hidden throughout the text.

Let's therefore introduce the maximal marginal relevance search.



In [12]:
retrived_doc =vectorstore.max_marginal_relevance_search(query = question,
                                           k=3, lambda_mult = 0.1 )# favoring document diversity

In [13]:
for i in retrived_doc:
    print(f"Page Contennt: {i.page_content} \n---------\n Lecture title:{i.metadata['Lecture Title']}\n")

Page Contennt: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching! 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: Such as using an analysis to explain how a story ended the way it did or how there was a decrease in 

In [17]:
#Let's instruct the method to retrieve chunks only from the second lecture in the doc file.

retrived_doc2 =vectorstore.max_marginal_relevance_search(query = question,
                                           k=3, lambda_mult = 0.1,
                                         filter = {"Lecture Title": '''Programming Languages & Software Employed in Data Science - All the Tools You Need'''}
) 
# favoring document diversity

In [19]:
for i in retrived_doc2:
    print(f"Page Contennt: {i.page_content} \n---------\n Lecture title:{i.metadata['Lecture Title']}\n")

Page Contennt: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: Among the many applications we have plotted, we can say there is an increasing amount of software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB. In terms of big data, Hadoop is the name that must stick with you. Hadoop is listed as a software in the sense that it is a collection of programs, but don’t imagine it as a nice-looking application 
---------
 Lecture title:Pr

In [22]:
retrived_doc3 =vectorstore.max_marginal_relevance_search(query = question,
                                           k=3, lambda_mult = 0.7,
                                         filter = {"Lecture Title": '''Programming Languages & Software Employed in Data Science - All the Tools You Need'''}
) 


In [23]:
for i in retrived_doc3:
    print(f"Page Contennt: {i.page_content} \n---------\n Lecture title:{i.metadata['Lecture Title']}\n")

Page Contennt: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations 
---------
 Lecture ti

In [25]:
#Finally, let's remove the effect of the diversity component altogether by setting the lambda parameter
retrived_doc4 =vectorstore.max_marginal_relevance_search(query = question,
                                           k=3, lambda_mult = 1,
                                         filter = {"Lecture Title": '''Programming Languages & Software Employed in Data Science - All the Tools You Need'''}
) 


In [27]:
for i in retrived_doc4:
    print(f"Page Contennt: {i.page_content} \n---------\n Lecture title:{i.metadata['Lecture Title']}\n")

#this was our result when we ran the similarity search method earlier in this lesson.
#A lambda value of one removes the diversity component altogether, leaving only relevance.



Page Contennt: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
---------
 Lecture title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Contennt: Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you ne